# B1 · Your First Prompts

*Path B: AI-Assisted, notebook 1 of 5 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Work in a loop: describe, generate, read, run, check
- Describe your data to an AI assistant without sharing the data itself
- Write a prompt that gets you usable code
- Check that the code did what you wanted

**Before this notebook:** the six Foundations notebooks, ending with [F6: Combining Tables](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/F6_Combining_Tables.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. A different way of working

In the Foundations notebooks you wrote every line yourself. From here on, an AI assistant writes most of the code. Your job changes: you **describe** what you want, **read** what comes back, and **check** that it is right.

That does not make the Foundations wasted effort. They are exactly what lets you do this well. You know what a dataframe is, what `filter()` and `left_join()` do, and why missing values matter. Without that, you could not tell good code from bad.

### Three rules

1. **Never give an AI tool participant-level data.** Under [*All of Us* policy](https://support.researchallofus.org/hc/en-us/articles/34814131370388), uploading to a public AI tool counts as disseminating the data, and moving individual-level data out of the Researcher Workbench violates the Data User Code of Conduct. The working rule in this course follows from that: you share code and descriptions of your data, never the data. Section 4 spells out what that means. *The data in these notebooks are simulated, so nothing here is restricted. Practice the habit anyway.*
2. **You are responsible for every line.** If you cannot explain what a line does, do not use it in your research. Ask the AI to explain it until you can.
3. **Check before you trust.** Code that runs without an error is not necessarily correct.


### Which AI assistant?

Use the AI chat tool your instructor specifies, and keep it open in a **separate window or tab** while you work.

Do not use an assistant that is built into the notebook itself (Colab has one, called Gemini). A built-in assistant may be able to read your cells and their output, which with real data would mean it sees the data without your pasting anything. With the simulated data here that would do no harm, but a separate chat window makes you practice describing your data, and that is the skill you will need.

## 2. The project

Over the next few notebooks you will carry out a small research project the way you would with *All of Us* data:

> **Is taking an anticholinergic medication associated with developing dementia?**

You have three tables of **simulated** data for 100 people. The outcome is recorded as either dementia or mild cognitive impairment (MCI), an earlier stage of memory loss. In this project both count, and the notebooks call them "the condition".

Before any analysis, the tables have to be cleaned and combined into one table with one row per person.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal.

In [ ]:
library(tidyverse)

person_df    <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/demographic_data.csv")
condition_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/conditions.csv")
drug_df      <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/drug_exposures.csv")

# ---- Helper code. You do not need to read this part; just run the cell. ----
describe_columns <- function(data) {
  tibble(column = names(data), type = map_chr(data, ~ class(.x)[1]))
}
check <- function(ok, label) {
  ok <- tryCatch(suppressWarnings(isTRUE(ok)), error = function(e) FALSE)
  cat(if (ok) "PASS    " else "NOT YET ", label, "\n")
}

## 3. The loop

Every task follows the same five steps.

| Step | What you do |
|---|---|
| 1. **Describe** | Tell the AI about your data and what you want |
| 2. **Generate** | The AI writes code |
| 3. **Read** | Go through the code. Can you say what each part does? |
| 4. **Run** | Paste it into the notebook and run it |
| 5. **Check** | Compare the result with something you know |

If step 3 or step 5 fails, go back to step 1 with a better description, or ask the AI to explain.

## 4. Describing data without sharing it

A chat assistant in a separate window cannot see your notebook, so it knows nothing about your data unless you tell it. The more precisely you describe the data, the better the code.

You might think of pasting the output of `head()`. With real data, **do not**: `head()` prints actual rows, and rows are participant-level data. The setup cell gave you a safer tool, `describe_columns()`, which prints only the column names and types.

In [ ]:
describe_columns(person_df)

This is safe to paste into any AI tool. (`numeric` is a number; `character` is text.)

For a column of categories, the AI also needs to know which categories exist. The names of the categories, with no counts and no rows, are fine to share.

In [ ]:
unique(person_df$race)

Use `unique()` this way **only on a column you know holds a short list of categories**, such as race or sex. On a column of IDs, dates, or free text, `unique()` prints values that belong to individual people.

### What may be shared

| Safe to give an AI tool | Never give an AI tool |
|---|---|
| Your code | Rows of data, even one |
| Column names and types (`describe_columns()`) | The output of `head()`, `str()`, or a printed dataframe |
| The names of the categories in a column | `person_id` values, dates, or anything else about one person |
| Error messages, once you have read them and removed any data values they quote | Screenshots or plots that show individual participants |
| Your research question and analysis plan | Tables or counts in which any cell is below 20 |
| Summary statistics with no participant count below 20 | |

Two consequences that are easy to miss:

- When a result looks wrong, **describe the problem to the AI in words**. Do not paste the output table.
- The assistant does not need to see your data to write good code. It needs to know how the data are laid out.

## 5. A worked example: recoding race

Follow this example through once. In the next section you will do one yourself.

**The task.** The `race` column has seven categories. For analysis, you want to shorten "Black or African American" to "Black", relabel "Skip" as "Unknown", and combine "More than one population" and "None of these" into "Other".

First, look at what you have, so that you can compare afterwards.

In [ ]:
person_df %>% count(race)

### Step 1: Describe

Here is a prompt for this task.

> In R using tidyverse, I have a dataframe called `person_df` with one row per person. It has a column `race` (character) with these values: "Asian", "Black or African American", "More than one population", "None of these", "Other", "Skip", "White".
>
> Change "Black or African American" to "Black". Change "Skip" to "Unknown". Change both "More than one population" and "None of these" to "Other". Leave all other values unchanged. Save the result back to `person_df`.

Notice what it contains:

| Part of the prompt | Why it is there |
|---|---|
| "In R using tidyverse" | So you get code in the language and style you know |
| "a dataframe called `person_df`" | So the code uses your real names and runs without editing |
| "one row per person" | So the AI understands the table's layout |
| "a column `race` (character) with these values..." | The AI cannot guess your categories or their exact spelling |
| The changes, one by one | No room for interpretation |
| "Leave all other values unchanged" | Says what should *not* happen |
| "Save the result back to `person_df`" | Says where the result should go |

### Step 2: Generate

An AI assistant returned the code in the cell below. **Do not run it yet.** (If you try the prompt yourself, your AI may write it differently. That is fine: there is more than one correct way.)

In [ ]:
person_df <- person_df %>%
  mutate(
    race = case_when(
      race == "Black or African American" ~ "Black",
      race == "Skip" ~ "Unknown",
      race %in% c("More than one population", "None of these") ~ "Other",
      .default = race
    )
  )

### Step 3: Read

Read the code before running it. You know `mutate()` from the Foundations. You have not seen `case_when()` before, and that is normal: AI assistants often use functions that are new to you. Here is how to read this one.

- `case_when()` is a list of rules. Each rule is `test ~ new value`. R uses the first rule whose test is true.
- `race %in% c("A", "B")` is true when race is "A" or race is "B".
- `.default = race` says what to do when no rule matches: keep the original value.

When you meet an unfamiliar function, **ask the AI**: *"Explain this code line by line for a beginner."* That is always a reasonable request.

### Step 4: Run

Now go back and run the cell.

### Step 5: Check

The code ran without an error. That tells you only that R understood it, not that it did what you wanted. Three checks for any recoding:

In [ ]:
person_df %>% count(race)   # 1. Are the categories what you intended?

In [ ]:
nrow(person_df)             # 2. Is the number of rows unchanged?

In [ ]:
sum(is.na(person_df$race))  # 3. Were any missing values created?

Five categories, 100 rows, no missing values. And the numbers add up: "Other" was 4 and is now 15, which is 4 + 6 + 5. The recoding worked.

## 6. Your turn, with help: recoding sex at birth

**The task.** Change every `sex_at_birth` value other than "Female" and "Male" to "Other". Save the result back to `person_df`.

Start by gathering what the AI will need to know.

In [ ]:
unique(person_df$sex_at_birth)

### Step 1: Describe

Here is a prompt with gaps. Copy it into your AI assistant and fill in the gaps.

> In R using tidyverse, I have a dataframe called `______` with one row per person. It has a column `______` (character) with these values: ______.
>
> Change ______ to "Other". Leave "Female" and "Male" unchanged. Save the result back to `______`.

**My prompt:** *(double-click this cell and paste the prompt you used)*

### Steps 2 to 4: Generate, read, run

Read the code the AI gives you. If any part is unclear, ask the AI to explain it. Then paste it below and run it.

Your data are already loaded. If the AI's answer includes lines that create example data, read a file, or install a package, leave those lines out.

In [ ]:
# Paste the AI's code here, read it, then run it

### Step 5: Check

First do your own check, using the three checks from the worked example.

In [ ]:
# Your own check

Then run the provided check. Each line prints `PASS` or `NOT YET`.

In [ ]:
check(nrow(person_df) == 100, "person_df still has 100 rows")
check(!anyNA(person_df$sex_at_birth), "no missing values were created")
check(setequal(unique(person_df$sex_at_birth), c("Female", "Male", "Other")), "there are exactly three categories")
check(sum(person_df$sex_at_birth == "Other") == 8, "8 people are coded Other")

If a line says `NOT YET`, something is off. Look at `person_df %>% count(sex_at_birth)`, work out what went wrong, and tell the AI in words what needs to change. If you have changed `person_df` in a way you did not intend, run the setup cell and the race recoding again to start fresh.

## 7. Weak prompts and strong prompts

Compare these two prompts for the task you just did.

**Weak:** "Clean up the sex variable."

**Strong:** the prompt you completed above.

The weak prompt forces the AI to guess the name of your dataframe, the name of the column, which values exist, and what "clean up" means. It will guess, and it will not tell you that it guessed. The code may run and still do something you did not intend.

## 8. The AI as a tutor

The same assistant that writes code can explain it. Useful questions:

- "Explain this code line by line for a beginner."
- "What does `.default` do here? What would happen if I left it out?"
- "Is there a simpler way to write this?"
- "What could go wrong with this code?"

Try the second question now, with the code you were given. The answer describes a real and common mistake that you will meet again in notebook B4.

---
## Recap

| Put this in your prompt | Why |
|---|---|
| R and tidyverse | You get code you can read |
| The dataframe's name | The code runs without editing |
| Column names and types (`describe_columns()`) | The AI cannot see your data |
| What one row represents | One row per person, or per record? |
| The exact category values | Spelling and capital letters must match |
| The goal, step by step | Leaves nothing to guess |
| What should stay unchanged | Prevents side effects |
| The name for the result | You know where to find it |
| That the data are already loaded | Stops the AI inventing example data that would overwrite yours |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Section 6: recoding sex at birth**

A completed prompt: *In R using tidyverse, I have a dataframe called `person_df` with one row per person. It has a column `sex_at_birth` (character) with these values: "Female", "Male", "Intersex", "Prefer not to answer", "Skip". Change "Intersex", "Prefer not to answer", and "Skip" to "Other". Leave "Female" and "Male" unchanged. Save the result back to `person_df`.* One correct answer:

```r
person_df <- person_df %>%
  mutate(
    sex_at_birth = case_when(
      sex_at_birth %in% c("Intersex", "Prefer not to answer", "Skip") ~ "Other",
      .default = sex_at_birth
    )
  )
```

---
## Next

Continue with [B2: Building a Dataset with AI](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B2_Building_a_Dataset_with_AI.ipynb).